# ROGII True-Start Failure Atlas: Tail Risk Before Modeling

The official metric pools squared error across rows. This means a few long, badly tracked wells can dominate many ordinary wells. Before adding PF, DTW, HMM, KNN surfaces, or boosters, it helps to quantify exactly where the simplest anchor fails and what kind of signal an advanced model must recover.

This notebook evaluates all 773 training wells at the supplied `TVT_input` boundary. It reports the last-visible-TVT baseline, the concentration of total squared error, and two diagnostic oracles fitted only on the hidden truth: a per-well constant and a per-well linear TVT path. The oracles are not deployable predictions; they separate level drift from nonlinear path error and provide a useful ceiling for candidate design.

No submission is generated and no leaderboard information is used.

In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CANDIDATES = [
    Path('/kaggle/input/competitions/rogii-wellbore-geology-prediction'),
    Path('/kaggle/input/rogii-wellbore-geology-prediction'),
    Path('data/raw/competition'),
    Path('../../data/raw/competition'),
]
DATA_ROOT = next((path for path in CANDIDATES if path.exists()), None)
if DATA_ROOT is None:
    raise FileNotFoundError('Competition data was not found')
paths = sorted((DATA_ROOT/'train').glob('*__horizontal_well.csv'))
print('Data root:', DATA_ROOT)
print('Training wells:', len(paths))

In [ ]:
records = []
for path in paths:
    frame = pd.read_csv(path, usecols=['MD','Z','GR','TVT','TVT_input'])
    observed = frame.TVT_input.notna().to_numpy()
    missing = np.flatnonzero(~observed)
    if len(missing) == 0 or missing[0] == 0 or observed[missing[0]:].any():
        raise ValueError(f'Invalid boundary in {path.name}')
    b = int(missing[0])
    md = frame.MD.to_numpy(float)[b:]
    truth = frame.TVT.to_numpy(float)[b:]
    anchor = float(frame.TVT_input.iloc[b-1])
    anchor_error = anchor-truth

    constant = np.full(len(truth), truth.mean())
    xc = md-md.mean()
    slope = float(xc@(truth-truth.mean()) / max(xc@xc, 1e-12))
    linear = truth.mean() + slope*xc
    records.append({
        'well_id': path.name.removesuffix('__horizontal_well.csv'),
        'n_rows': len(frame), 'n_eval': len(truth), 'eval_fraction': len(truth)/len(frame),
        'z_span': float(np.ptp(frame.Z.to_numpy(float)[b:])),
        'target_min_delta': float((truth-anchor).min()),
        'target_max_delta': float((truth-anchor).max()),
        'target_span': float(np.ptp(truth)),
        'anchor_sse': float(anchor_error@anchor_error),
        'constant_oracle_sse': float((constant-truth)@(constant-truth)),
        'linear_oracle_sse': float((linear-truth)@(linear-truth)),
        'suffix_linear_slope': slope,
    })
wells = pd.DataFrame(records)
for candidate in ['anchor','constant_oracle','linear_oracle']:
    wells[f'{candidate}_rmse'] = np.sqrt(wells[f'{candidate}_sse']/wells.n_eval)
print('Evaluation rows:', int(wells.n_eval.sum()))

## Baseline and diagnostic oracle gap

The constant oracle knows the mean TVT of the real suffix, so its gap from anchor measures reducible level drift. The linear oracle also knows the suffix slope, so its remaining error measures curvature and local geological deviations. These are diagnostic lower bounds only; any real method must infer them from prefix, trajectory, GR, typewell, and neighbors.

In [ ]:
summary_rows = []
for name in ['anchor','constant_oracle','linear_oracle']:
    x = wells[f'{name}_rmse']
    summary_rows.append({
        'candidate': name,
        'pooled_rmse': math.sqrt(wells[f'{name}_sse'].sum()/wells.n_eval.sum()),
        'macro_rmse': x.mean(), 'median': x.median(),
        'p90': x.quantile(.90), 'p95': x.quantile(.95), 'worst': x.max(),
    })
summary = pd.DataFrame(summary_rows).set_index('candidate')
display(summary.style.format('{:.3f}'))

## How concentrated is pooled squared error?

Ranking wells by anchor SSE shows how much of the official-style loss comes from the tail. This is why candidate reports should include per-well p95/worst and SSE concentration, not only a single pooled score.

In [ ]:
ranked = wells.sort_values('anchor_sse', ascending=False).reset_index(drop=True)
ranked['cumulative_sse_share'] = ranked.anchor_sse.cumsum()/ranked.anchor_sse.sum()
concentration = {}
for fraction in [0.01,0.05,0.10,0.20]:
    count = max(1, int(math.ceil(len(ranked)*fraction)))
    concentration[f'top {fraction:.0%} wells'] = ranked.anchor_sse.iloc[:count].sum()/ranked.anchor_sse.sum()
display(pd.Series(concentration, name='share of anchor SSE').to_frame().style.format('{:.1%}'))
display(ranked.head(15)[['well_id','n_eval','anchor_rmse','target_span','z_span','suffix_linear_slope','cumulative_sse_share']])

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes[0,0].hist(wells.anchor_rmse, bins=40, color='#2563eb', alpha=.85)
axes[0,0].axvline(wells.anchor_rmse.quantile(.95), color='#dc2626', linestyle='--', label='p95')
axes[0,0].set(title='Anchor per-well RMSE', xlabel='RMSE', ylabel='wells'); axes[0,0].legend()
axes[0,1].scatter(wells.n_eval, wells.anchor_rmse, s=10, alpha=.4, c=wells.target_span, cmap='viridis')
axes[0,1].set(title='Length is not the only difficulty', xlabel='evaluation rows', ylabel='anchor RMSE')
axes[1,0].scatter(wells.target_span, wells.anchor_rmse, s=10, alpha=.4, c=wells.suffix_linear_slope, cmap='coolwarm')
axes[1,0].set(title='Target motion drives anchor error', xlabel='suffix TVT span', ylabel='anchor RMSE')
axes[1,1].plot(np.arange(1,len(ranked)+1)/len(ranked), ranked.cumulative_sse_share, color='#7c3aed')
axes[1,1].plot([0,1],[0,1],'k--',linewidth=1)
axes[1,1].set(title='Anchor SSE concentration', xlabel='fraction of wells (worst first)', ylabel='cumulative SSE share')
plt.tight_layout(); plt.show()

## Modeling ideas implied by the atlas

- **Level correction is high value.** The constant-oracle gap quantifies how much can be gained by estimating only the suffix mean displacement from anchor. A conservative residual model may capture this before a full row-wise tracker.
- **Path dynamics still matter.** The linear-oracle gap isolates nonlinear residual structure; GR/typewell alignment, PF/HMM, and spatial surfaces should be judged on this remaining component.
- **Use an anchor safety prior.** A tracker that wins ordinary wells but creates a few large wrong branches can lose on pooled RMSE. Fixed or uncertainty-aware anchor blending is a natural defense.
- **Stratify validation.** Report results by evaluation length, target span, `Z` span, prefix GR calibration quality, and tracker disagreement.
- **Keep gates prefix-only.** Target span and oracle values in this notebook are diagnostic labels. They must never become test-time gate features.

A useful candidate should reduce both total SSE concentration and p95 error. If it only shifts errors between wells, a better pooled mean on a small sample may not transfer.